# Préparation des données – Contrats automobiles chez ENSAssuRances

ROY Tao

2026-03-29

## Introduction
Ce document présente le processus complet de préparation et de nettoyage des données issues de la base Contrat.

L’objectif de cette étape est de produire un jeu de données propre, cohérent et exploitable pour les analyses statistiques.

Les principales étapes de préparation sont :

- import et inspection initiale des données

- rennomage des colonnes

- nettoyage du contenu des variables qualitatives

- nettoyage du contenu des variables quantitatives

- Conversion des dates et tests de cohérences

- Détections et suppression des doublons

- analyse et traitement des valeurs manquantes

- imputation de variables critiques

## 1. Chargement des librairies

Les bibliothèques suivantes sont utilisées pour la manipulation, la visualisation et le nettoyage des données :

In [35]:
import pandas as pd
from itables import show
from pandas.api.types import CategoricalDtype

## 2. Importation des données

Les données sont importées depuis un fichier xlsx.

In [9]:
df_brut = pd.read_excel("data/Contrat.xlsx")

On fait une inspection de la table chargée :

In [10]:
show(df_brut.head(100), 
     scrollX=True, 
     lengthMenu=[5, 10, 25, 50], 
     pageLength=5,
     classes="cell-border stripe")

Loading ITables v2.7.3 from the internet... (need help?)


In [11]:
print(f"Dimensions : {df_brut.shape}") # (Lignes, Colonnes)
print("-" * 30)
df_brut.info()
display(df_brut.head(3).T) # Aperçu transposé des 3 premières lignes

Dimensions : (301437, 40)
------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 301437 entries, 0 to 301436
Data columns (total 40 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   idxCt                 301437 non-null  str    
 1   idxYear               301437 non-null  int64  
 2   vhImmat               301437 non-null  str    
 3   sitStartDate          301437 non-null  str    
 4   sitEndDate            301437 non-null  str    
 5   sitExpo               301437 non-null  float64
 6   drv1Age               301437 non-null  float64
 7   drv1Sex               301437 non-null  str    
 8   drv1DriveLicenceType  301437 non-null  str    
 9   drv1DriveLicenceAge   301437 non-null  float64
 10  vhAge                 301437 non-null  float64
 11  ctFrm                 301437 non-null  str    
 12  ctAssBase             301437 non-null  int64  
 13  ctAss0km              301437 non-null  int64  
 14  ctAssV

,0,1,2
idxCt,C002513884,C002513884,C002513884
idxYear,2019,2020,2021
vhImmat,ME-6556-LY,ME-6556-LY,ME-6556-LY
sitStartDate,2019-01-01,2020-01-01,2021-01-01
sitEndDate,2019-12-31,2020-12-31,2021-12-31
sitExpo,1.0,1.0,1.0
drv1Age,39.52,40.52,41.52
drv1Sex,H,H,H
drv1DriveLicenceType,Cond Accompagnée,Cond Accompagnée,Cond Accompagnée
drv1DriveLicenceAge,18.68,19.68,20.68


## 3. Rennomage des colonnes

Pour une utilisation plus simple de la donnée on va modifier le nom des colonnes comme ceci :

In [ ]:
# Création du dictionnaire complet pour les 40 variables de la table Contrats
data_contrats_complet = {
    "Nom d'origine": [
        "idxCt", "idxYear", "vhImmat", "sitStartDate", "sitEndDate", "sitExpo", "drv1Age", "drv1Sex", 
        "drv1DriveLicenceType", "drv1DriveLicenceAge", "vhAge", "ctFrm", "ctAssBase", "ctAss0km", 
        "ctAssVHR", "vhSegment", "vhMarque", "vhEnergy", "vhWeight", "vhDIN", "vhValue", "vhGroup", 
        "vhClass", "ctUsage", "ctKM", "ctDeduc", "claimsAnt", "ctINSEE", "id1_AssBase", "id1_Ass0km", 
        "id1_AssVHR", "id2_AssBase", "id2_Ass0km", "id2_AssVHR", "id3_AssBase", "id3_Ass0km", 
        "id3_AssVHR", "COT_AssBase", "COT_Ass0km", "COT_AssVHR"
    ],
    "Nouveau nom": [
        "id_contrat", "annee_exercice", "immatriculation", "date_debut_sit", "date_fin_sit", "exposition", "age_conducteur", "genre_conducteur",
        "permis_type", "anciennete_permis", "age_vehicule", "formule_contrat", "garantie_base", "garantie_0km",
        "garantie_vhr", "vehicule_segment", "vehicule_marque", "vehicule_energie", "vehicule_poids", "puissance_din", "valeur_vehicule", "vehicule_groupe",
        "vehicule_classe", "usage_contrat", "option_petit_rouleur", "franchise_montant", "nb_sinistres_ant", "code_insee_commune", "id1_sinistre_base", "id1_sinistre_0km",
        "id1_sinistre_vhr", "id2_sinistre_base", "id2_sinistre_0km", "id2_sinistre_vhr", "id3_sinistre_base", "id3_sinistre_0km",
        "id3_sinistre_vhr", "cotisation_base", "cotisation_0km", "cotisation_vhr"
    ],
    "Type": [
        "texte", "entier", "texte", "texte", "texte", "decimal", "decimal", "texte", 
        "texte", "decimal", "decimal", "texte", "entier", "entier", 
        "entier", "texte", "texte", "texte", "entier", "entier", "decimal", "texte", 
        "texte", "texte", "texte", "entier", "entier", "texte", "texte", "texte", 
        "texte", "texte", "texte", "texte", "texte", "texte", 
        "texte", "decimal", "decimal", "decimal"
    ],
    "Description": [
        "Identifiant contrat", "Exercice", "Immatriculation du véhicule", "Date de début de la situation", "Date de fin de la situation", "Exposition de la situation", "Age du conducteur principal", "Genre du conducteur principal",
        "Mode d'obtention du permis", "Ancienneté du permis", "Age du véhicule", "Formule souscrite au contrat", "Assistance de Base (1/0)", "Assistance 0 km (1/0)",
        "Assistance Véhicule de Remplacement (1/0)", "Segment commercial du véhicule", "Marque du véhicule", "Alimentation du véhicule", "Poids du véhicule", "Puissance DIN du véhicule", "Valeur du véhicule", "Groupe du véhicule",
        "Classe du véhicule", "Usage déclaré au contrat", "Option Petit Rouleur", "Franchise souscrite", "Nombre de sinistres antérieurs", "Code INSEE de la commune", "ID sinistre Assistance Base n°1", "ID sinistre Assistance 0km n°1",
        "ID sinistre Assistance VHR n°1", "ID sinistre Assistance Base n°2", "ID sinistre Assistance 0km n°2", "ID sinistre Assistance VHR n°2", "ID sinistre Assistance Base n°3", "ID sinistre Assistance 0km n°3",
        "ID sinistre Assistance VHR n°3", "Cotisation payée Assistance Base", "Cotisation payée Assistance 0km", "Cotisation payée Assistance VHR"
    ]
}

df_dico_complet = pd.DataFrame(data_contrats_complet)

# Affichage élégant
print("DICTIONNAIRE COMPLET DES DONNÉES - CONTRATS (40 VARIABLES)")
display(df_dico_complet)

DICTIONNAIRE COMPLET DES DONNÉES - CONTRATS (40 VARIABLES)


,Nom d'origine,Nouveau nom,Type,Description
0,idxCt,id_contrat,texte,Identifiant contrat
1,idxYear,annee_exercice,entier,Exercice
2,vhImmat,immatriculation,texte,Immatriculation du véhicule
3,sitStartDate,date_debut_sit,texte,Date de début de la situation
4,sitEndDate,date_fin_sit,texte,Date de fin de la situation
5,sitExpo,exposition,decimal,Exposition de la situation
6,drv1Age,age_conducteur,decimal,Age du conducteur principal
7,drv1Sex,genre_conducteur,texte,Genre du conducteur principal
8,drv1DriveLicenceType,type_permis,texte,Mode d'obtention du permis
9,drv1DriveLicenceAge,anciennete_permis,decimal,Ancienneté du permis


In [47]:
# Dictionnaire de renommage strict basé sur le dictionnaire des données 
mapping_contrats = {
    "idxCt": "id_contrat",
    "idxYear": "annee_exercice",
    "vhImmat": "immatriculation",
    "sitStartDate": "date_debut_sit",
    "sitEndDate": "date_fin_sit",
    "sitExpo": "exposition",
    "drv1Age": "age_conducteur",
    "drv1Sex": "genre_conducteur",
    "drv1DriveLicenceType": "type_permis",
    "drv1DriveLicenceAge": "anciennete_permis",
    "vhAge": "age_vehicule",
    "ctFrm": "formule_contrat",
    "ctAssBase": "garantie_base",
    "ctAss0km": "garantie_0km",
    "ctAssVHR": "garantie_vhr",
    "vhSegment": "vehicule_segment",
    "vhMarque": "vehicule_marque",
    "vhEnergy": "vehicule_energie",
    "vhWeight": "vehicule_poids",
    "vhDIN": "puissance_din",
    "vhValue": "valeur_vehicule",
    "vhGroup": "groupe_vehicule",
    "vhClass": "classe_vehicule",
    "ctUsage": "usage_contrat",
    "ctKM": "option_km",
    "ctDeduc": "franchise",
    "claimsAnt": "nb_sinistres_ant",
    "ctINSEE": "code_insee",
    "id1_AssBase": "id1_sinistre_base",
    "id1_Ass0km": "id1_sinistre_0km",
    "id1_AssVHR": "id1_sinistre_vhr",
    "id2_AssBase": "id2_sinistre_base",
    "id2_Ass0km": "id2_sinistre_0km",
    "id2_AssVHR": "id2_sinistre_vhr",
    "id3_AssBase": "id3_sinistre_base",
    "id3_Ass0km": "id3_sinistre_0km",
    "id3_AssVHR": "id3_sinistre_vhr",
    "COT_AssBase": "cotisation_base",
    "COT_Ass0km": "cotisation_0km",
    "COT_AssVHR": "cotisation_vhr"
}

# Application du renommage
df_prep = df_brut.rename(columns=mapping_contrats)

Nouveaux noms des 40 colonnes :

In [20]:
df_prep.columns

Index(['id_contrat', 'annee_exercice', 'immatriculation',
       'date_debut_situation', 'date_fin_situation', 'exposition_risque',
       'age_conducteur', 'genre_conducteur', 'permis_type',
       'anciennete_permis', 'age_vehicule', 'formule_contrat',
       'garantie_assistance_base', 'garantie_assistance_0km',
       'garantie_assistance_vhr', 'vehicule_segment', 'vehicule_marque',
       'vehicule_energie', 'vehicule_poids', 'vehicule_puissance_din',
       'vehicule_valeur', 'vehicule_groupe', 'vehicule_classe',
       'usage_contrat', 'option_petit_rouleur', 'franchise_montant',
       'nb_sinistres_anterieurs', 'code_insee_commune', 'id1_sinistre_base',
       'id1_sinistre_0km', 'id1_sinistre_vhr', 'id2_sinistre_base',
       'id2_sinistre_0km', 'id2_sinistre_vhr', 'id3_sinistre_base',
       'id3_sinistre_0km', 'id3_sinistre_vhr', 'cotisation_base',
       'cotisation_0km', 'cotisation_vhr'],
      dtype='str')

## 4. Nettoyage du contenu des variables qualitatives

### 4.1 Normalisation de la colonne immatriculation

In [25]:
# Nombre total de lignes vs nombre d'immatriculations uniques
total_lignes = len(df_prep)
uniques = df_prep['immatriculation'].nunique()

print(f"Total lignes : {total_lignes}")
print(f"Immatriculations uniques : {uniques}")

Total lignes : 301437
Immatriculations uniques : 69686


On regarde si toute les plaques sont bien noté au bon format :

In [24]:
# On cherche ce qui ne ressemble pas à du texte avec des tirets
format_incorrect = df_prep[~df_prep['immatriculation'].str.contains(r'[A-Z]{2}-\d+-?[A-Z]{2}', na=False, regex=True)]

print(f"Nombre de lignes au format suspect : {len(format_incorrect)}")
display(format_incorrect['immatriculation'].head())

Nombre de lignes au format suspect : 0


Series([], Name: immatriculation, dtype: str)

### 4.2 Normalisation de la colonne genre_conducteur

In [26]:
df_prep['genre_conducteur'].value_counts()

genre_conducteur
H    180200
F    121237
Name: count, dtype: int64

On voit que l'on a bien nos 2 catégories cependant on va changer le nom des valeurs pour qu'elles soient plus explicite :

In [27]:
mapping_genre = {
    "H": "Homme",
    "F": "Femme"
}

# Application de la transformation
df_prep['genre_conducteur'] = df_prep['genre_conducteur'].replace(mapping_genre)

# Vérification du nouveau "table()"
print(df_prep['genre_conducteur'].value_counts())

genre_conducteur
Homme    180200
Femme    121237
Name: count, dtype: int64


On transforme aussi la variable en catégorie.

In [31]:
df_prep['genre_conducteur'] = df_prep['genre_conducteur'].astype('category')

### 4.3 Normalisation de la colonne permis_type

In [29]:
df_prep['permis_type'].value_counts()

permis_type
Traditionnel        226920
Cond Accompagnée     74517
Name: count, dtype: int64

On voit que l'on a bien nos 2 catégories cependant on va changer le nom des valeurs pour qu'elles soient plus explicite :

In [30]:
mapping_type = {
    "Traditionnel": "Traditionnel",
    "Cond Accompagnée": "Conduite Accompagnée"
}

# Application de la transformation
df_prep['permis_type'] = df_prep['permis_type'].replace(mapping_type)

# Vérification du nouveau "table()"
print(df_prep['permis_type'].value_counts())

permis_type
Traditionnel            226920
Conduite Accompagnée     74517
Name: count, dtype: int64


On transforme aussi la variable en catégorie.

In [32]:
df_prep['permis_type'] = df_prep['permis_type'].astype('category')

### 4.4 Normalisation de la colonne formule_contrat

In [33]:
df_prep['formule_contrat'].value_counts()

formule_contrat
Mini    192558
Maxi     59441
Med1     25480
Med2     23958
Name: count, dtype: int64

On a bien toute nos catégories que l'on transorme en catégorie et que l'on place dans l'ordre.

In [36]:
niveau_formules = CategoricalDtype(
    categories=['Mini', 'Med1', 'Med2', 'Maxi'], 
    ordered=True
)

# On applique la transformation
df_prep['formule_contrat'] = df_prep['formule_contrat'].astype(niveau_formules)

### 4.5 Normalisation de la colonne vehicule_segment

In [37]:
df_prep['vehicule_segment'].value_counts()

vehicule_segment
Citadine     120614
Familiale     75566
Compacte      59953
SUV           45304
Name: count, dtype: int64

On transforme la variable en catégorie.

In [38]:
df_prep['vehicule_segment'] = df_prep['vehicule_segment'].astype('category')

### 4.6 Normalisation de la colonne vehicule_marque

In [39]:
df_prep['vehicule_marque'].value_counts()

vehicule_marque
Renault          60970
Citroen          60461
Peugeot          59382
Autre            29975
Nissan            9353
Seat              9231
Audi              9148
Mercedes Benz     9099
Toyota            9079
Dacia             9053
Fiat              8995
BMW               8987
Ford              8870
Volkswagen        8834
Name: count, dtype: int64

On transforme la variable en catégorie.

In [40]:
df_prep['vehicule_marque'] = df_prep['vehicule_marque'].astype('category')

### 4.7 Normalisation de la colonne vehicule_energie

In [41]:
df_prep['vehicule_energie'].value_counts()

vehicule_energie
Essence               135559
Diesel                120352
Electrique/Hybride     45526
Name: count, dtype: int64

On transforme la variable en catégorie.

In [42]:
df_prep['vehicule_marque'] = df_prep['vehicule_marque'].astype('category')

### 4.9 Normalisation de la colonne classe_vehicule

In [52]:
df_prep['vehicule_classe'].value_counts()

vehicule_classe
3    141403
2     94569
4     54106
5      6512
1      4847
Name: count, dtype: int64

On transforme la variable en catégorie.

In [53]:
df_prep['vehicule_classe'] = df_prep['vehicule_classe'].astype('category')

### 4.10 Normalisation de la colonne usage_contrat

In [54]:
df_prep['usage_contrat'].value_counts()

usage_contrat
Pri    241594
Pro     59843
Name: count, dtype: int64

On voit que l'on a bien nos 2 catégories cependant on va changer le nom des valeurs pour qu'elles soient plus explicite :

In [55]:
mapping_usage = {
    "Pri": "Privé",
    "Pro": "professionnelle"
}

# Application de la transformation
df_prep['usage_contrat'] = df_prep['usage_contrat'].replace(mapping_usage)

# Vérification du nouveau "table()"
print(df_prep['usage_contrat'].value_counts())

usage_contrat
Privé              241594
professionnelle     59843
Name: count, dtype: int64


On transforme la variable en catégorie.

In [56]:
df_prep['usage_contrat'] = df_prep['usage_contrat'].astype('category')

### 4.11 Normalisation de la colonne option_petit_rouleur

In [58]:
df_prep['option_petit_rouleur'].value_counts()

option_petit_rouleur
N    260973
O     40464
Name: count, dtype: int64

On voit que l'on a bien nos 2 catégories cependant on va changer le nom des valeurs pour qu'elles soient plus explicite :

In [59]:
mapping_option = {
    "N": "Non",
    "O": "Oui"
}

# Application de la transformation
df_prep['option_petit_rouleur'] = df_prep['option_petit_rouleur'].replace(mapping_option)

# Vérification du nouveau "table()"
print(df_prep['option_petit_rouleur'].value_counts())

option_petit_rouleur
Non    260973
Oui     40464
Name: count, dtype: int64


On transforme la variable en catégorie.

In [61]:
df_prep['option_petit_rouleur'] = df_prep['option_petit_rouleur'].astype('category')

## 5. Nettoyage du contenu des variables quantitatives